# 3.5-B · JOINs, CTEs & Window Functions
### Financial Analytics — Module 3.5 (PostgreSQL)

The three techniques that separate "knows some SQL" from "can work in a finance team". Same setup cell as 3.5-A:

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

PG_URL = os.environ.get("COURSE_DB_URL", "")
if PG_URL:
    engine = create_engine(PG_URL)
    print("Connected to course PostgreSQL")
else:
    engine = create_engine("sqlite://")
    BASE = "data/"
    pd.read_csv(BASE + "client_book.csv").to_sql("clients", engine, index=False)
    pd.read_csv(BASE + "messy_transactions.csv").to_sql("transactions", engine, index=False)
    print("Local practice database built from CSVs")

def q(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

---
## JOIN — combining tables

Notebook 3C's `merge`, in its native habitat. `clients.client_id` matches `transactions.customer_id`.

In [ ]:
q("""
    SELECT
        c.segment,
        COUNT(t.txn_id)              AS transactions,
        ROUND(SUM(t.amount_inr), 0)  AS total_spend
    FROM clients c
    LEFT JOIN transactions t
        ON c.client_id = t.customer_id
    WHERE t.amount_inr > 0 OR t.amount_inr IS NULL   -- keep non-transactors!
    GROUP BY c.segment
    ORDER BY total_spend DESC
""")

Reading it: `c` and `t` are table nicknames (*aliases*); `ON` is the join key; **LEFT JOIN keeps every client** even with zero transactions — exactly the survivorship decision from 3C. An `INNER JOIN` would silently drop non-transactors.

| pandas | SQL |
|---|---|
| `how="left"` | `LEFT JOIN` |
| `how="inner"` | `INNER JOIN` (or just `JOIN`) |
| `how="outer"` | `FULL OUTER JOIN` |

### ✏️ Exercise 1
Per **city** (from the *clients* table, the trustworthy one): number of distinct transacting clients and total spend. Hint: `COUNT(DISTINCT t.customer_id)`.

In [ ]:
q("""
    -- your query here
    SELECT 1
""")

---
## CTEs — WITH clauses that keep queries readable

A CTE (Common Table Expression) names an intermediate result, like assigning a DataFrame to a variable. Anything beyond one step should use them.

In [ ]:
q("""
    WITH spend_per_client AS (
        SELECT customer_id, SUM(amount_inr) AS total_spend
        FROM transactions
        WHERE amount_inr > 0
        GROUP BY customer_id
    )
    SELECT
        c.segment,
        ROUND(AVG(s.total_spend), 0) AS avg_spend_of_transactors,
        COUNT(*)                     AS transacting_clients
    FROM spend_per_client s
    JOIN clients c ON c.client_id = s.customer_id
    GROUP BY c.segment
    ORDER BY avg_spend_of_transactors DESC
""")

Step 1 (the CTE) aggregates transactions; step 2 joins and summarises. Each step is testable alone — run just the inside of the `WITH` to verify it. That's the tracer-bullet habit in SQL.

---
## Window functions — the interview separator

A window function computes across related rows **without collapsing them** — every row keeps its identity but gains context. GROUP BY answers "total per segment"; a window answers "**each client's rank within their segment**".

In [ ]:
q("""
    SELECT segment, client_id, aum_inr, rnk FROM (
        SELECT
            segment, client_id, aum_inr,
            RANK() OVER (PARTITION BY segment ORDER BY aum_inr DESC) AS rnk
        FROM clients
        WHERE aum_inr IS NOT NULL
    ) ranked
    WHERE rnk <= 3
    ORDER BY segment, rnk
""")

Top-3 clients per segment, one query. `PARTITION BY` = "restart per group", `ORDER BY` = ranking order. The pattern `RANK() OVER (PARTITION BY ... ORDER BY ...)` is the single most-asked SQL interview construct in analytics roles.

In [ ]:
# Running totals - the same OVER idea with SUM
q("""
    SELECT
        segment,
        client_id,
        aum_inr,
        SUM(aum_inr) OVER (PARTITION BY segment ORDER BY aum_inr DESC) AS running_aum
    FROM clients
    WHERE segment = 'Ultra-HNI' AND aum_inr IS NOT NULL
    LIMIT 10
""")

Running totals answer concentration questions instantly: *how much of the segment's AUM sits in its top 5 clients?* The pandas twins are `rank()`, `cumsum()`, and `rolling()` — window functions are their SQL family.

### ✏️ Exercise 2
Using a window function, find each **city's** single largest client by AUM. (Pattern: rank within city partition, keep `rnk = 1`.)

### ✏️ Exercise 3 — the five business questions
Answer each in SQL, then verify in pandas (load the CSVs and reproduce the numbers — they must match *exactly*):
1. Which segment generates the most transactions per client?
2. What share of total AUM does each city hold, as a percentage? (Hint: `SUM(aum_inr) OVER ()` with no partition = grand total.)
3. Which merchant has the highest average ticket size (amounts > 0 only)?
4. How many clients have an SIP but zero transactions in this file?
5. Among churned clients, which risk profile is over-represented vs its share of all clients?

In [ ]:
# workspace for exercises 2-3


---
## Recap — the full Rosetta Stone

| Idea | pandas | SQL |
|---|---|---|
| Join tables | `merge(..., how="left")` | `LEFT JOIN ... ON` |
| Named intermediate step | `tmp = df.groupby(...)` | `WITH tmp AS (...)` |
| Rank within group | `groupby().rank()` | `RANK() OVER (PARTITION BY ...)` |
| Running total | `cumsum()` | `SUM(x) OVER (ORDER BY ...)` |
| Grand-total share | `x / x.sum()` | `x / SUM(x) OVER ()` |

**When to use which:** SQL to *get and pre-shape* the data where it lives; pandas to *explore, iterate and chart* once it's yours. Professionals fluently do both in the same hour.

**Module 3.5 complete.** Claim the **Query Fluent** badge.

---
*AI disclosure: ______*